In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.compose import ColumnTransformer

df = pd.read_csv("titanic_data_updated.csv")

In [3]:
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
71,72,no,third,"Goodwin, Miss. Lillian Amy",female,16.0,5,2,CA 2144,46.9000,NaN,S
564,565,no,third,"Meanwell, Miss. (Marion Ogden)",female,NaN,0,0,SOTON/O.Q. 392087,8.0500,NaN,S
34,35,no,first,"Meyer, Mr. Edgar Joseph",male,28.0,1,0,PC 17604,82.1708,NaN,C
512,513,yes,first,"McGough, Mr. James Robert",male,36.0,0,0,PC 17473,26.2875,E25,S
105,106,no,third,"Mionoff, Mr. Stoytcho",male,28.0,0,0,349207,7.8958,NaN,S


Deleting useless columns

In [4]:
df.drop(['PassengerId','Name','Ticket'],axis=1,inplace=True)

Combining 'SibSp' and 'Parch' columns

In [5]:
df['FamilyMembers'] = df['SibSp'] + df['Parch'] + 1 # 1 is for the person him/herself

In [6]:
df.drop(columns=['SibSp','Parch'],inplace=True)

In [7]:
df.sample(2)

,Survived,Pclass,Sex,Age,Fare,Cabin,Embarked,FamilyMembers
306,yes,first,female,NaN,110.8833,NaN,C,1
676,no,third,male,24.5,8.0500,NaN,S,1


Train Test Split

In [8]:
x = df.drop(columns='Survived') # features
y = df['Survived'] # target

x_train, x_test, y_train, y_test = train_test_split(x,y,train_size=0.2, random_state=42)

In [9]:
x_test.sample(2)

,Pclass,Sex,Age,Fare,Cabin,Embarked,FamilyMembers
602,first,male,NaN,42.4000,NaN,S,1
704,third,male,26.0,7.8542,NaN,S,2


Imputation

In [10]:
imputer_transformer = ColumnTransformer(
    transformers=[
        ('age',SimpleImputer(missing_values=np.nan,strategy='mean'),['Age']),
        ('embarked',SimpleImputer(missing_values=np.nan,strategy='most_frequent'),['Embarked']),
        ('cabin',SimpleImputer(missing_values=np.nan,strategy='constant',fill_value='Missing',add_indicator=True),['Cabin'])
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)

imputer_transformer.set_output(transform='pandas')

# fit
imputer_transformer.fit(x_train)

# transform
x_train = imputer_transformer.transform(x_train)
x_test = imputer_transformer.transform(x_test)

In [11]:
x_train.isnull().sum()

Age                       0
Embarked                  0
Cabin                     0
missingindicator_Cabin    0
Pclass                    0
Sex                       0
Fare                      0
FamilyMembers             0
dtype: int64

In [12]:
x_test.isnull().sum()

Age                       0
Embarked                  0
Cabin                     0
missingindicator_Cabin    0
Pclass                    0
Sex                       0
Fare                      0
FamilyMembers             0
dtype: int64

In [13]:
x_train.sort_values(by='Age').tail(10)

,Age,Embarked,Cabin,missingindicator_Cabin,Pclass,Sex,Fare,FamilyMembers
698,49.0,C,C68,False,first,male,110.8833,3
52,49.0,C,D33,False,first,female,76.7292,2
458,50.0,S,Missing,True,second,female,10.5000,1
406,51.0,S,Missing,True,third,male,7.7500,1
492,55.0,S,C30,False,first,male,30.5000,1
647,56.0,C,A26,False,first,male,35.5000,1
366,60.0,C,D37,False,first,female,75.2500,2
170,61.0,S,B19,False,first,male,33.5000,1
555,62.0,S,Missing,True,first,male,26.5500,1
252,62.0,S,C87,False,first,male,26.5500,1


### Outliers Handling

Age Outliers

In [14]:
mean_age = x_train['Age'].mean()
std_age = x_train['Age'].std()

x_train['z_score_age'] = (x_train['Age']-mean_age)/std_age

outliers_age = x_train[abs(x_train['z_score_age'])>3]

print(f"Number of outliers: {len(outliers_age)}")
# display(outliers_age)
# x_train.sort_values(by='Age').tail(10)

Number of outliers: 0


Fare outliers

In [15]:
mean_fare = x_train['Fare'].mean()
std_fare = x_train['Fare'].std()

x_train['z_score_fare'] = (x_train['Fare']-mean_fare)/std_fare
outliers_fare = x_train[abs(x_train['z_score_fare'])>3]

print(f"Number of outlier fare: {len(outliers_fare)}")
display(outliers_fare.head())

Number of outlier fare: 3


,Age,Embarked,Cabin,missingindicator_Cabin,Pclass,Sex,Fare,FamilyMembers,z_score_age,z_score_fare
27,19.0,S,C23 C25 C27,False,first,male,263.000,6,-0.981541,6.267455
498,25.0,S,C22 C26,False,first,female,151.550,4,-0.462901,3.280644
700,18.0,C,C62 C64,False,first,female,227.525,2,-1.067981,5.316741


Fare outliers using IQR

In [16]:
fare_q1 = x_train['Fare'].quantile(0.25)
fare_q3 = x_train['Fare'].quantile(0.75)

fare_iqr = fare_q3 - fare_q1

max_range = fare_q3 + (1.5 * fare_iqr)
min_range = max(0,fare_q1 - (1.5 * fare_iqr)) # becasue the value goes to negative side, so capping at 0

print(f"Max: {max_range}\nMin: {min_range}")

fare_outliers = x_train[(x_train['Fare']<min_range) | (x_train['Fare'] > max_range)]
print("Number of fare outliers: ",len(fare_outliers))
display(fare_outliers.sample(5))

Max: 65.3438
Min: 0
Number of fare outliers:  24


,Age,Embarked,Cabin,missingindicator_Cabin,Pclass,Sex,Fare,FamilyMembers,z_score_age,z_score_fare
52,49.0,C,D33,False,first,female,76.7292,2,1.611659,1.275479
1,38.0,C,C85,False,first,female,71.2833,2,0.660819,1.129531
681,27.0,C,D49,False,first,male,76.7292,1,-0.290021,1.275479
504,16.0,S,B79,False,first,female,86.5000,1,-1.240861,1.537332
484,25.0,C,B49,False,first,male,91.0792,2,-0.462901,1.660053


In [17]:
# omitting outliers age from the x_train data
x_train = x_train[abs(x_train['z_score_age']) <=3]
x_train

,Age,Embarked,Cabin,missingindicator_Cabin,Pclass,Sex,Fare,FamilyMembers,z_score_age,z_score_fare
761,41.000000,S,Missing,True,third,male,7.1250,1,0.920139,-0.589883
645,48.000000,C,D33,False,first,male,76.7292,2,1.525219,1.275479
754,48.000000,S,Missing,True,second,female,65.0000,4,1.525219,0.961141
556,48.000000,C,A16,False,first,female,39.6000,2,1.525219,0.280433
850,4.000000,S,Missing,True,third,male,31.2750,7,-2.278141,0.057326
...,...,...,...,...,...,...,...,...,...,...
106,21.000000,S,Missing,True,third,female,7.6500,1,-0.808661,-0.575814
270,30.355172,S,Missing,True,first,male,31.0000,1,0.000000,0.049956
860,41.000000,S,Missing,True,third,male,14.1083,3,0.920139,-0.402734
435,14.000000,S,B96 B98,False,first,female,120.0000,4,-1.413741,2.435117


In [18]:
# Capping 'Fare' at 0 to max range of IQR outliers
x_train['Fare'] = x_train['Fare'].clip(min_range,max_range)
print(x_train['Fare'].min())
print(x_train['Fare'].max())

0.0
65.3438


In [17]:
# x_train.sort_values(by='Age').tail(5)
x_train.sort_values(by='Fare').tail(5)

,Age,Embarked,Cabin,missingindicator_Cabin,Pclass,Sex,Fare,FamilyMembers,z_score_age,z_score_fare
681,27.0,C,D49,False,first,male,65.3438,1,-0.290021,1.275479
385,18.0,S,Missing,True,second,male,65.3438,1,-1.067981,1.188938
700,18.0,C,C62 C64,False,first,female,65.3438,2,-1.067981,5.316741
435,14.0,S,B96 B98,False,first,female,65.3438,4,-1.413741,2.435117
102,21.0,S,D26,False,first,male,65.3438,2,-0.808661,1.290441


In [19]:
x_train = x_train.drop(columns=['z_score_age', 'z_score_fare'], errors='ignore')
x_test = x_test.drop(columns=['z_score_age', 'z_score_fare'], errors='ignore')

In [20]:
x_train.sample(2)

,Age,Embarked,Cabin,missingindicator_Cabin,Pclass,Sex,Fare,FamilyMembers
724,27.0,S,E8,False,first,male,53.100,2
216,27.0,S,Missing,True,third,female,7.925,1


### Encoding and Scaling

In [21]:
# making a column with the first Character of Cabin name which is 'deck' 
x_train['Cabin_deck'] = x_train['Cabin'].astype(str).str[0];
x_test['Cabin_deck'] = x_test['Cabin'].astype(str).str[0];

In [22]:
x_train.sample(2)

,Age,Embarked,Cabin,missingindicator_Cabin,Pclass,Sex,Fare,FamilyMembers,Cabin_deck
80,22.0,S,Missing,True,third,male,9.0,1,M
726,30.0,S,Missing,True,second,female,21.0,4,M


In [23]:
encoder_scaler = ColumnTransformer(
    transformers= [
        ('pclass',OrdinalEncoder(categories=[['third','second','first']]),['Pclass']),
        ('sex_embarked_cabin',OneHotEncoder(sparse_output=False,drop='first'),['Embarked','Sex','Cabin_deck']),
        ('age',StandardScaler(),['Age']),
        ('fare_family',MinMaxScaler(),['Fare','FamilyMembers'])
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)

encoder_scaler.set_output(transform='pandas')

encoder_scaler.fit(x_train)

x_train = encoder_scaler.transform(x_train)
x_test = encoder_scaler.transform(x_test)

In [24]:
x_train.drop('Cabin',axis=1,inplace=True)
x_test.drop('Cabin',axis=1,inplace=True)

In [25]:
x_train.sample(2)

,Pclass,Embarked_Q,Embarked_S,Sex_male,Cabin_deck_B,Cabin_deck_C,Cabin_deck_D,Cabin_deck_E,Cabin_deck_F,Cabin_deck_G,Cabin_deck_M,Cabin_deck_T,Age,Fare,FamilyMembers,missingindicator_Cabin
13,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.749367,0.478622,0.6,True
524,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.000000,0.110633,0.0,True


In [26]:
x_test.sample(2)

,Pclass,Embarked_Q,Embarked_S,Sex_male,Cabin_deck_B,Cabin_deck_C,Cabin_deck_D,Cabin_deck_E,Cabin_deck_F,Cabin_deck_G,Cabin_deck_M,Cabin_deck_T,Age,Fare,FamilyMembers,missingindicator_Cabin
403,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.204155,0.242563,0.1,True
220,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-1.244361,0.123195,0.0,True
